In [3]:
import logging

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lower, split, explode

from pyspark.sql import SparkSession
from pyspark.sql.functions import split, explode, lower, regexp_replace, col, transform

from pyspark.sql import DataFrame, SparkSession
from pyspark.sql.types import StructType, StructField, DoubleType, StringType
from pyspark.sql import functions as F



METERS_PER_FOOT = 0.3048
FEET_PER_MILE = 5280
EARTH_RADIUS_IN_METERS = 6371e3  # 6,371,000 meters
METERS_PER_MILE = METERS_PER_FOOT * FEET_PER_MILE
EARTH_RADIUS_IN_KM =  EARTH_RADIUS_IN_METERS / 1000

In [2]:
# create or get existing SparkSession
spark = SparkSession.builder \
    .appName("notebook_session") \
    .master("local[*]") \
    .config("spark.ui.showConsoleProgress", "false") \
    .getOrCreate()

logging.getLogger("py4j").setLevel(logging.WARNING)
logging.info("SparkSession ready: %s", spark)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/12/31 00:21:38 WARN Utils: Your hostname, codespaces-3eed66, resolves to a loopback address: 127.0.0.1; using 10.0.1.208 instead (on interface eth0)
25/12/31 00:21:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/31 00:21:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [19]:
df = spark.read.csv("/workspaces/dataengineer-transformations-python/resources/word_count/words_output", header=True)

In [ ]:
def haversine_distance(col_lat1, col_lon1, col_lat2, col_lon2):
    """
    Compute Haversine distance between two (lat, lon) points in meters.
    Returns a Column expression usable inside withColumn().
    """
    lat1 = F.radians(col_lat1)
    lon1 = F.radians(col_lon1)
    lat2 = F.radians(col_lat2)
    lon2 = F.radians(col_lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (
        F.sin(dlat / 2) ** 2 +
        F.cos(lat1) * F.cos(lat2) * F.sin(dlon / 2) ** 2
    )

    c = 2 * F.asin(F.sqrt(a))

    return EARTH_RADIUS_IN_KM * c

+---+------------------+-------------------+----------+----------+
|id |random_double     |random_normal      |random_int|random_str|
+---+------------------+-------------------+----------+----------+
|0  |44.31972728415477 |54.28826246482898  |536       |43408581  |
|1  |78.7964074278653  |29.687284600968724 |906       |16b9dd7b  |
|2  |85.21532394169246 |34.441041127615826 |110       |d208588c  |
|3  |27.188775318163817|76.84304975271671  |948       |6e3abccc  |
|4  |84.37469941022493 |-24.464153611657547|240       |371f5d3a  |
|5  |2.544925223302208 |5.126015401067552  |252       |1cbf0f0b  |
|6  |70.82985654738927 |-37.31424096275175 |918       |0af94c05  |
|7  |63.636421769602826|-2.4009404772278544|469       |4035f5ff  |
|8  |37.39066690381295 |-80.19954836993011 |118       |1112693e  |
|9  |9.107074022872673 |-69.64630408278626 |829       |08ed225c  |
+---+------------------+-------------------+----------+----------+
only showing top 10 rows
